# Convergence population — **batched** driver (cardio-only, SI stack)

<details>
<summary>Integrates the whole sampled population as **one vmapped solve per calibration stage**</summary>

on the step-independent (SI) stack — instead of the serial per-sample loop in
`Convergence_Run.ipynb`. Toggle CPU/GPU and float64/float32 in the imports cell.

- Batching is the real lever (BATCH_PLAN.md): it speeds up the sweep **even on CPU**.
- Final-state-only solvers keep peak memory at `(N, nState)` (no dense trajectories),
  so large `nrModels` fit a 6 GB card; very large N is chunked via `chunkSize`.
- Solver is selectable (`euler`/`rk4`); both are pure `lax.scan` (vmappable).

</details>

In [ ]:
# region -> runConfig — the single run-configuration surface (device/precision applied before JAX)
# The ONE place run configuration lives (project rule); defined first so device/precision applies before JAX inits.
runConfig = {
    # --- file references ---
    "model":    "cvModel_linear.json",   # config/models/ cardio-only model (16 calibration controllers)
    "scenario": "sepsis_linear.json",    # config/scenarios/ twin + calibration + convergence
    "mode":     "calibration",    # staged calibration run per population member

    # --- pipeline phases ---
    "run":  False,   # phase 1 — run the LHS sweep + save the artifact
    "plot": True,   # phase 2 — load + analyse + plot

    # --- device / precision (applied in Imports cell, before `import jax`) ---
    "device": {
        "useGpu":    False,       # True -> CUDA device
        "precision": "float64",   # "float64" or "float32"
    },

    # --- population sweep ---
    "population": {
        "nrModels":    1024,        # members in the batched solve
        "seed":        0,         # LHS seed
        "errorTarget": 0.5,       # SUCCESS if max |rel err| (%) <= this
    },

    # --- batched solve ---
    "solver":    {"type": "euler"}, # "rk4" or "euler"
    "chunkSize": 1024,                # samples per vmap (VRAM bound)

    # --- calibration overrides (per-key merge over scenario calibration; {} = as-is) ---
    "calibration": {},

    # --- analysis ---
    "analysis": {
        "atm":             760.0,     # atmospheric offset for absolute-pressure signals
        "divergenceLimit": 50000.0,   # |value| >= this in any obs/param -> BAD run
    },

    # --- plotting ---
    "plotOpts": {
        "targetPoints": 1500,   # decimation target for the full-raw fallback
        "tailSeconds":  3000.0,  # zoom plots: window (simulated s) at the END of the calibration
    },

    "progressEvery": 10,   # emit live convergence line every N simulated seconds (0 = off)

    "saveRaw":   True,   # stream every saved run's full trajectory to disk

    # --- output ---
    "output": {
        "save": True,
        "path": "notebookData/convergence",
        "name": "population_linear_batch.h5",
        "logProgress": True,   # persist live-progress trace into the .h5
    },

    "postProcessing": None,   # off: population file is the artifact
    "requested":      None,   # off
    "plots":          [],     # off
    "printStatus":    True,   # print chunk progress
    "printEveryPct":  100,    # print every this % of chunks

    # --- integration numerics (override scenario shared.integration) ---
    "runTime": 10,       # simulated seconds per internal run
    "dt":      0.0005,   # integrator step
    "dtDense": 1.0,      # save/output grid: 1.0 = 1 Hz; raise for within-beat waveforms
}
# endregion

## Imports

In [ ]:
# region -> imports + device/precision (must precede `import jax`)
# ---- repo-root bootstrap: run from any cwd (make `library` importable + resolve
# ---- the relative notebookData/ + config/ paths). Walks up to the dir containing library/. ----
import os, sys
_root = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(_root, "library")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
os.chdir(_root)

# ---- device / precision (from runConfig, MUST run before JAX initialises) -------
useGpu    = runConfig["device"]["useGpu"]
precision = runConfig["device"]["precision"]

import os
if useGpu:
    os.environ.pop("CUDA_VISIBLE_DEVICES", None)
    os.environ["JAX_PLATFORMS"] = "cuda"
    # GPU memory hygiene (must precede `import jax`): grow on demand instead of grabbing
    # ~75% of VRAM up front (so JAX coexists with the display on a small shared card), and
    # hand freed buffers back to the driver so the cleanup cell / del actually releases VRAM.
    os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
    os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"]   = "platform"
else:
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
    os.environ["JAX_PLATFORMS"] = "cpu"

import jax
jax.config.update("jax_enable_x64", precision == "float64")

import numpy as np
from scipy.stats import qmc                      # Latin Hypercube sampling
import pandas as pd
import matplotlib.pyplot as plt
import json
import time

import library.run.runner as runner          # buildSimulationParams (+ calibratorUpdater)
import library.run.stateSetup as stateSetup   # resolveCalibrationBounds (single-source calibration ranges)
import library.run.runnerBatchSI as runnerBatchSI   # batched (vmapped) SI calibration
import library.viz.plots as libPlots         # plotCalibrationConvergence
import library.utils as utils
from library.hdf5 import schema_pop                       # population artifact (init_population / final_states)
from library.hdf5.raw_stream import RawTraceStreamWriter  # async streaming writer for the full raw tensor
import library.postproc.reporting as reporting            # shared scope / rejection report

np.set_printoptions(suppress=True)
print("devices:", jax.devices(), "| x64:", jax.config.jax_enable_x64)
# endregion

## Assemble `simulationParams` + load the convergence config

<details>
<summary>`runner.buildSimulationParams` expands the slim `runConfig` + scenario into the legacy</summary>

`simulationParams` shape. The `convergence` block of the scenario carries the parameter
bounds (LHS) and the observation list (the controllers' `varTarget`s + `V_Vs`).

</details>

In [ ]:
# region -> assemble simulationParams + load the convergence config (params, bounds, observations)
scenario = utils.loadScenario(runConfig["scenario"])
simulationParams = runner.buildSimulationParams(runConfig, scenario)

model         = utils.loadJSONfile(utils.configPath("models", runConfig["model"]))
conv          = scenario["convergence"]
calConf       = simulationParams["simulationConf"]["calibration"]
twin          = scenario["shared"]["twin"]["twinTargets"]
volDist       = scenario["shared"]["twin"]["volumeDistribution"]
param_names   = calConf["adaptive"]["parameters"]                                    # swept set = calibration params
bounds        = stateSetup.resolveCalibrationBounds(model["calibration"], calConf, param_names)  # (P, 2) single source
observations  = conv["observations"]
pop           = runConfig["population"]
outPath       = os.path.join(runConfig["output"]["path"], runConfig["output"]["name"])

print(f"nrModels = {pop['nrModels']} | params = {len(param_names)} | observations = {len(observations)}")
print(f"output -> {outPath}")
# endregion

## Twin-target array + atmospheric offsets

<details>
<summary>Each observation has a target derived from the twin state: pressures from the twin pressure</summary>

targets, stroke volume from `CO/HR`, compartment volumes from `volumeDistribution ×
TotalBloodVolume`, and `Cyc_HC` from `60/HR`. Absolute-pressure observations carry the model's
+760 mmHg atmospheric offset, which we subtract to compare in gauge mmHg (amplitudes, volumes
and SV carry no offset).

</details>

In [ ]:
# region -> twin-target array + atmospheric offsets per observation
ATM = runConfig["analysis"]["atm"]

def obsOffset(name):
    """Atmospheric offset baked into absolute-pressure signals (gauge = raw - offset)."""
    return utils.obsOffset(name, ATM)

def obsTarget(name):
    """Twin target for an observation (gauge units), or None if untargeted.

    Matches Convergence_Population.ipynb's targetsArray: the capillary means are
    derived as a pressure DROP from the upstream arterial target, not as absolute
    means -- avg_P_Cs = (Sys_P_As - amp_P_As) - twin.avg_P_Cs (systemic diastolic
    minus the configured drop), avg_P_Cp = Dia_P_Ap - twin.avg_P_Cp.
    """
    TBV = twin["TotalBloodVolume"]
    direct = {
        "avg_P_Vs": twin["CVP"],
        "avg_P_Cs": twin["Sys_P_As"] - twin["amp_P_As"] - twin["avg_P_Cs"],
        "avg_P_Cp": twin["Dia_P_Ap"] - twin["avg_P_Cp"],
        "keep_max_P_As": twin["Sys_P_As"], "keep_max_P_Ap": twin["Sys_P_Ap"],
        "keep_min_P_Ap": twin["Dia_P_Ap"], "amp_P_As": twin["amp_P_As"],
        # pulmonary PP mirrors progress.obsTargetArrays: explicit twin amp_P_Ap wins, else Sys - Dia
        "amp_P_Ap": twin.get("amp_P_Ap", twin["Sys_P_Ap"] - twin["Dia_P_Ap"]),
        "keep_SV_Hl": twin["CO"] / twin["HR"], "Cyc_HC": 60.0 / twin["HR"],
        "V_Vs": volDist["Vs"] * TBV,
    }
    if name in direct:
        return direct[name]
    if name.startswith("avg_V_"):
        return volDist[name[len("avg_V_"):]] * TBV
    return None

targetArr = np.array([obsTarget(o) if obsTarget(o) is not None else np.nan for o in observations])
offsetArr = np.array([obsOffset(o) for o in observations])
pd.DataFrame({"observation": observations, "target": targetArr, "offset": offsetArr})
# endregion

## Latin Hypercube sample of the parameter space

<details>
<summary>`sampled_params` is an `(nrModels, P)` matrix scaled into each parameter's `[min, max]`</summary>

bounds. Each row becomes the initial value of the corresponding controlled state for one
population member (`runner.run(..., stateOverrides=row)`).

</details>

In [ ]:
# region -> Latin Hypercube sample of the parameter space
if runConfig["run"]:
    sampler = qmc.LatinHypercube(d=len(param_names), seed=pop["seed"])
    unit = sampler.random(n=pop["nrModels"])                      # (N, P) in [0, 1)
    sampled_params = qmc.scale(unit, bounds[:, 0], bounds[:, 1])   # (N, P) in [min, max]
    print("sampled_params shape:", sampled_params.shape)
    pd.DataFrame(sampled_params, columns=param_names).head()
# endregion

## Run the population (single batched / vmapped solve + async raw streaming)

<details>
<summary>One `runnerBatchSI.batchedCalibration` call replaces the per-sample Python loop: the population</summary>

is integrated in **sample chunks** (`chunkSize`, which bounds VRAM), each chunk running the full
stage stack on device. Diverged lanes propagate non-finite values (no per-sample try/except) and
are masked to `BAD_RUN_SENTINEL`.

With `saveRaw=True`, **every saved run's full trajectory** (all states + algebraic outputs) is
streamed to the HDF5 `raw` dataset `(N, T_total, C)` by a background writer thread — the
compressed disk write overlaps the next chunk's GPU compute, so RAM/VRAM stay bounded by
`chunkSize` while the full (tens-of-GB) raw tensor lands on disk. The file is created up front
(`init_population`) so the writer can append `raw` immediately.

</details>

In [ ]:
# region -> run the batched (vmapped) population solve + async raw streaming, save the artifact
if runConfig["run"]:
    if runConfig["output"]["save"]:
        os.makedirs(runConfig["output"]["path"], exist_ok=True)

    # Converged-window signals for the calibration plot (obs + swept params).
    traceNames = list(dict.fromkeys(list(observations) + list(param_names)))

    # --- set up the population file up front (init_population must precede the raw writer) ---
    # One scalar prep is shared by the file setup and the run.
    prep = runnerBatchSI.prepare(simulationParams)
    layout = runnerBatchSI.rawLayout(simulationParams, prepared=prep)
    stateNames = layout["stateNames"]
    rawDtype = "float64" if jax.config.jax_enable_x64 else "float32"

    saveRaw = runConfig.get("saveRaw", False) and runConfig["output"]["save"]
    if runConfig["output"]["save"]:
        schema_pop.init_population(
            outPath, param_names=param_names, state_names=stateNames,
            observation_names=observations, sampled_params=sampled_params,
            model_structure=utils.modelStructureJSON(prep["modelStructure"]),
            problem={"names": param_names, "bounds": bounds.tolist(), "num_vars": len(param_names)},
            conf=runConfig, meta={"twinTargets": twin})

    def rawWriterFactory(signalNames, totalPoints, time_, nDense):
        print(f"  raw writer: ({pop['nrModels']}, {totalPoints}, {len(signalNames)}) {rawDtype} "
              f"+ gzip  (~{pop['nrModels'] * totalPoints * len(signalNames) * (8 if rawDtype=='float64' else 4) / 1e9:.1f} GB uncompressed)")
        return RawTraceStreamWriter(outPath, N=pop["nrModels"], signalNames=signalNames,
                                    totalPoints=totalPoints, nDense=nDense, time=time_, dtype=rawDtype)

    t0 = time.time()
    batch = runnerBatchSI.batchedCalibration(
        simulationParams, sampled_params, param_names, observations,
        chunkSize=runConfig.get("chunkSize", 64), printStatus=runConfig.get("printStatus", True),
        printEveryPct=runConfig.get("printEveryPct"), traceNames=traceNames,
        rawWriterFactory=(rawWriterFactory if saveRaw else None), prepared=prep)
    totalWall = time.time() - t0
    print(f"Batched population complete in {totalWall:.1f}s "
          f"(N={pop['nrModels']}, solver={simulationParams['solver']['type']}, "
          f"x64={jax.config.jax_enable_x64}, saveRaw={saveRaw}) -> {outPath}")

    finalStates = batch["finalStates"]                 # (N, nState)
    modelStructure = batch["modelStructure"]
    traces, traceT = batch["traces"], batch["traceT"]  # {name: (N, nT)}, (nT,)

    # Gauge observation matrix (raw final value minus atmospheric offset == serial steadyState).
    obsMatrix = batch["rawObs"] - offsetArr            # (N, nObs)

    # Out-of-scope mask: a lane whose observation OR swept/calibrated parameter went non-finite or
    # left scope is a BAD run. Set those rows to NaN so the error-summary good-mask (below) drops
    # them exactly as the serial path does. The parameters ride the final-state columns.
    sIdxFS      = {n: i for i, n in enumerate(stateNames)}
    paramMatrix = np.column_stack([finalStates[:, sIdxFS[p]] for p in param_names if p in sIdxFS])
    finiteRow = schema_pop.good_run_mask(obsMatrix, runConfig["analysis"].get("divergenceLimit", 1e6),
                                         param_matrix=paramMatrix)
    obsMatrix[~finiteRow] = np.nan
    print(f"converged lanes: {finiteRow.sum()}/{pop['nrModels']}")

    # --- final_states table (NN/Table-5); full traces already streamed to top-level `raw` -----
    # write_final_states writes the table in one shot WITHOUT creating runs/{id} groups (the raw
    # tensor is the per-run store, so empty runs/{id}/raw groups would only mislead).
    if runConfig["output"]["save"]:
        fsArr = finalStates.copy()
        fsArr[~finiteRow] = schema_pop.BAD_RUN_SENTINEL
        schema_pop.write_final_states(outPath, fsArr, [str(i) for i in range(pop["nrModels"])])
        print(f"final_states written -> {outPath}  | raw tensor "
              f"{(pop['nrModels'], layout['totalSavedPoints'], len(layout['signalNames']))}"
              f"  (read via schema_pop.raw_trace / f['raw'])")
        # Batched wall clock: one vmapped solve, so total wall (+ amortized total/N) is the
        # meaningful cost -- no per-run distribution. Stored as a 1-element timings array.
        schema_pop.write_timings(outPath, [totalWall], meta={
            "device":     "gpu" if useGpu else "cpu",
            "precision":  precision,
            "solver":     simulationParams["solver"]["type"],
            "dt":         simulationParams["dt"],
            "runTime":    simulationParams["runTime"],
            "nrModels":   pop["nrModels"],
            "stack":      "SI",
            "total_wall": totalWall,
        })
        # persist the live-progress trace (the convergence lines the solve printed) for comparison
        schema_pop.write_progress(outPath, batch.get("progress"), meta={
            "model": runConfig["model"], "scenario": runConfig["scenario"],
            "mode": runConfig["mode"], "solver": simulationParams["solver"]["type"],
            "nrModels": pop["nrModels"], "stack": "SI"})
# endregion

# Phase 2 — Load & analyse (from the saved file)

<details>
<summary>Everything below reconstructs from the population `.h5` — no in-session run state</summary>

is required. After a kernel restart you can run the setup cells (config →
`simulationParams` → targets → LHS, all scenario-only and cheap) then the **load**
cell below and every analysis/plot/table cell, without re-running the sweep.

The load cell rebuilds `modelStructure` (from the stored `model_structure` JSON) and
`obsMatrix` / `finiteRow` (each observation's converged value = last saved `raw`
timepoint minus its atmospheric offset), plus the batch timings (`timingMeta`)
written in Phase 1. The full per-run trajectories stay on disk in the `raw` tensor,
which the convergence plots read directly.

</details>

In [ ]:
# region -> load everything the analysis needs straight from the saved population file
if runConfig["plot"]:
    # --- load everything the analysis below needs, straight from the saved file ----------
    # Reconstructs the in-memory run state (modelStructure, obsMatrix, finiteRow) so Phase 2 is
    # independent of Phase 1. The batch stores one top-level `raw` (N, T, C) tensor (+ optional
    # `raw_coarse` companion); obsMatrix is each observation's converged value = last raw
    # timepoint minus its atmospheric offset. Full trajectories stay on disk for the plots.
    import h5py

    with h5py.File(outPath, "r") as f:
        modelStructure = json.loads(f["model_structure"].asstr()[()])
        sig = list(f["raw_signal_names"].asstr()[:])
        src = "raw_coarse" if "raw_coarse" in f else "raw"
        lastRow = np.asarray(f[src][:, -1, :])            # (N, C) converged (run-end) values

    sigIdx = {n: i for i, n in enumerate(sig)}
    Nrun = lastRow.shape[0]
    obsMatrix = np.full((Nrun, len(observations)), np.nan)
    for j, o in enumerate(observations):
        if o in sigIdx:
            obsMatrix[:, j] = lastRow[:, sigIdx[o]] - obsOffset(o)

    # run-end swept/calibrated-parameter values -> checked for out-of-scope alongside observations
    # (a run whose observation stays finite but whose parameter blows up is still a BAD run).
    paramInScope = [p for p in param_names if p in sigIdx]
    paramMatrix  = (np.column_stack([lastRow[:, sigIdx[p]] for p in paramInScope])
                    if paramInScope else np.zeros((Nrun, 0)))

    finiteRow = schema_pop.good_run_mask(obsMatrix, runConfig["analysis"].get("divergenceLimit", 1e6),
                                         param_matrix=paramMatrix)
    obsMatrix[~finiteRow] = np.nan
    traces, traceT, saveRaw = None, None, True            # plots read the file (raw/raw_coarse) below

    runWall, timingMeta = schema_pop.read_timings(outPath)
    print(f"loaded {Nrun} runs from {outPath} | converged lanes {finiteRow.sum()}/{Nrun} "
          f"| timings: {'yes' if timingMeta else 'none'}")
# endregion

## Scope / rejection report

<details>
<summary>Breaks down which runs were dropped and **why**, before the error summary works on the survivors.</summary>

A run is out of scope when **any** observation *or* swept/calibrated parameter reaches `|value| >=
divergenceLimit` (`runConfig.analysis.divergenceLimit`), or the lane is NaN/Inf or sentinel-stamped.
Reports the total kept/dropped split, the per-reason counts (a run can trip more than one), and the
individual signals that drove the out-of-scope drops with their worst run-end magnitude.

</details>

In [ ]:
# region -> scope / rejection report (why each run was dropped)
if runConfig["plot"]:
    lim    = runConfig["analysis"].get("divergenceLimit", 1e6)
    # rawObs rebuilt un-blanked from the run-end tensor slice (obsMatrix is blanked in load).
    rawObs = np.column_stack([
        (lastRow[:, sigIdx[o]] - obsOffset(o)) if o in sigIdx else np.full(Nrun, np.nan)
        for o in observations])
    rep = reporting.scopeRejectionReport(rawObs, paramMatrix, observations, paramInScope, lim=lim)
# endregion

## Error summary

<details>
<summary>Filter faulty runs (any observation == `BAD_RUN_SENTINEL`), then compute per-observation</summary>

relative error `(obs - target) / target × 100` and absolute error against the twin targets.
A run is SUCCESS when `max |relative error| <= errorTarget`.

</details>

In [ ]:
# region -> error summary — drop faulty runs, per-observation relative/absolute error
if runConfig["plot"]:
    targeted = ~np.isnan(targetArr)                     # observations that have a twin target
    good = schema_pop.good_run_mask(obsMatrix, runConfig["analysis"].get("divergenceLimit", 1e6),
                                    param_matrix=paramMatrix)
    print(f"{good.sum()}/{len(good)} valid runs ({(~good).sum()} faulty/diverged dropped)")

    obsGood = obsMatrix[good][:, targeted]
    tgt = targetArr[targeted]
    errRel = (obsGood - tgt) / tgt * 100.0
    errAbs = obsGood - tgt
    obsTargeted = [o for o, t in zip(observations, targeted) if t]

    success = np.max(np.abs(errRel), axis=1) <= pop["errorTarget"]
    print(f"converged (max|rel err| <= {pop['errorTarget']}%): {success.sum()}/{len(success)}")

    summary = pd.DataFrame({
        "observation": obsTargeted,
        "target": tgt,
        "mean_rel_err_%": np.nanmean(errRel, axis=0),
        "std_rel_err_%": np.nanstd(errRel, axis=0),
        "mean_abs_err": np.nanmean(errAbs, axis=0),
    })
    summary
# endregion

## Plots

In [ ]:
# region -> boxplot: relative error distribution per observation
if runConfig["plot"]:
    # --- boxplot: relative error distribution per observation ---------------------------
    if good.sum() > 0:
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.boxplot(errRel, tick_labels=utils.labelsFor(obsTargeted, "latex"), showfliers=False)
        ax.axhline(0.0, color="k", lw=0.8)
        ax.axhline(pop["errorTarget"], color="r", ls="--", lw=0.8, label=f"±{pop['errorTarget']}% target")
        ax.axhline(-pop["errorTarget"], color="r", ls="--", lw=0.8)
        ax.set_ylabel("relative error (%)")
        ax.set_title(f"Convergence error across {good.sum()} valid runs")
        ax.tick_params(axis="x", rotation=90)
        ax.legend()
        plt.tight_layout()
        plt.show()
# endregion

## LaTeX summary table

<details>
<summary>Builds the Table-5-style summary (each targeted observation paired with the parameter that</summary>

controls it), rendered with paper-quality names from `config/labels.json` via
`utils.labelFor`, and written to `notebookData/convergence_summary.tex`.

</details>

In [ ]:
# region -> Table-5-style LaTeX summary
if runConfig["plot"]:
    # --- Table-5-style LaTeX summary -----------------------------------------------------
    # Each targeted observation paired with the calibration parameter that controls it
    # (controller varTarget -> param), with paper-quality names from config/labels.json.
    # Emitted in the manuscript's own table shape (booktabs brace group + \captionof), the
    # same form as tab:mcmcSummary / tab:gdSummary, so it drops straight into main.tex.
    # Targets with no controlling parameter (V_Vs, Cyc_HC) are left out, so the table is the
    # 16 observation/parameter pairs the paper's tab:summaryTable carries.
    calib = modelStructure["calibration"]
    paramForObs = {calib[p]["params"]["varTarget"]: p
                   for p in param_names if p in calib}        # observation -> controlling param

    arrFS, stateNames = schema_pop.final_states_array(outPath)  # (N, S) calibrated final states
    sIdx = {n: i for i, n in enumerate(stateNames)}
    paramVals = {p: arrFS[good, sIdx[p]] for p in param_names if p in sIdx}

    def sci(x):
        """One notation for every std cell, 4 chars wide: 3e-4, 2e-2. Spreads run 1e-5 to 1e-2,
        so a fixed-decimal format would print 0.000 for most of the parameter column."""
        return f"{x:.0e}".replace("e-0", "e-").replace("e+0", "e+")

    rows = {}
    for j, o in enumerate(obsTargeted):
        pv = paramVals.get(paramForObs.get(o, ""))
        if pv is None or pv.size == 0:
            continue                                          # untargeted-by-a-parameter row
        rows[utils.labelFor(o, "latex")] = [
            f"{tgt[j]:.4g}",
            f"{summary['mean_rel_err_%'][j]:.3f}",
            sci(summary['std_rel_err_%'][j]),
            utils.labelFor(paramForObs[o], "latex"),
            f"{np.mean(pv):.3f}",
            sci(np.std(pv)),
        ]

    latexTable = utils.generateLatexTableInline(
        rows,
        ["Vars.", "Targets", "Relative Error \\%", "std", "Param.", "Value", "std"],
        ref="tab:summaryTable",
        colSpec="C{0.7cm} C{0.7cm} |C{1.0cm} C{0.7cm}|C{0.7cm} C{0.7cm} C{0.7cm}",
        fontSize="small",
        unbreakable=False,
        caption=(
            f"Summary of the convergence test across {good.sum()} simulations with random initial "
            f"parameter values and volume distributions. For each calibration target, the table "
            f"reports the prescribed target value, the mean signed relative error and its standard "
            f"deviation at convergence, together with the corresponding calibrated parameter values "
            f"and their variability across runs."))

    out_tex = os.path.join(runConfig["output"]["path"], "convergence_summary.tex")
    with open(out_tex, "w") as f:
        f.write(latexTable)
    print(latexTable)
    print(f"\nLaTeX table written to: {out_tex}")
# endregion


## Calibration convergence plot

<details>
<summary>Port of the legacy `buildConvPlot`: one panel per **targeted observation**, overlaying each</summary>

population member's trajectory over the **whole calibration** against its twin target (dashed).

Reads the tiny **`raw_coarse`** companion (one converged value per saved run) — instant. If a
file has only the full `raw` tensor, it falls back to a single strided pass (slow: decompresses
the whole tensor); build the companion once with `schema_pop.build_raw_coarse(outPath, nDense=…)`.
Only converged runs are drawn, capped at `MAX_RUNS` lines per panel.

</details>

In [ ]:
# region -> calibration convergence: per-run observation traces vs target
if runConfig["plot"]:
    if good.sum() > 0:
        # --- calibration convergence: per-run trajectory over the WHOLE calibration vs target ----
        # Prefer the tiny `raw_coarse` companion (one converged value per saved run) -> instant.
        # Else fall back to a single strided pass over the full `raw` tensor (slow: decompresses it
        # all). Else (no raw on disk) use the in-memory converged window. To add raw_coarse to an
        # older file: schema_pop.build_raw_coarse(outPath)  (one-time, slow once).
        import h5py

        calib = modelStructure["calibration"]
        paramForObs = {calib[p]["params"]["varTarget"]: p for p in param_names if p in calib}
        targetsByObs = {o: t for o, t in zip(observations, targetArr)}
        offsetsByObs = {o: off for o, off in zip(observations, offsetArr)}
        goodIdx = np.where(finiteRow)[0]
        targetPoints = runConfig["plotOpts"]["targetPoints"] # strided-decimation target for the full-raw fallback

        goodTraces, plotT, titleScope = {}, traceT, "converged window"
        if saveRaw and runConfig["output"]["save"]:
            with h5py.File(outPath, "r") as f:
                if "raw_coarse" in f:                                   # fast path
                    src, tkey, ds, titleScope = "raw_coarse", "raw_coarse_time", 1, "whole calibration"
                else:                                                   # slow fallback (full raw)
                    src, tkey = "raw", "raw_time"
                    ds, titleScope = max(1, f["raw"].shape[1] // targetPoints), "whole calibration (full raw)"
                sig = list(f["raw_signal_names"].asstr()[:])
                names = [o for o in obsTargeted if o in sig]
                block = f[src][goodIdx, ::ds, :]                        # good runs only, one pass
                plotT = f[tkey][::ds]
            goodTraces = {o: block[:, :, sig.index(o)] for o in names}

        libPlots.plotCalibrationConvergence(
            goodTraces or {o: traces[o][finiteRow] for o in obsTargeted if o in (traces or {})},
            traceT=plotT, targets=targetsByObs, offsets=offsetsByObs,
            paramForObs=paramForObs,
            title=f"Calibration convergence ({titleScope}, {simulationParams['solver']['type']})")
        plt.show()
# endregion

## Calibration convergence plot — tail zoom

<details>
<summary>The same observation panels, zoomed to the last `plotOpts.tailSeconds` simulated seconds.</summary>

The whole-calibration view is dominated by the early transient, which flattens the converged
band against the target. This cell masks the time axis of the traces the cell above already
loaded (no extra disk read) to the tail window, so the residual spread around each twin target
is readable. Set `runConfig["plotOpts"]["tailSeconds"]` to change the window.

</details>

In [ ]:
# region -> calibration convergence (zoom): observations over the last `plotOpts.tailSeconds`
if runConfig["plot"]:
    if good.sum() > 0:
        # --- same traces as the plot above, masked to the END of the calibration --------------
        # Reuses `goodTraces` / `plotT` from the cell above (raw_coarse -> strided raw -> in-memory
        # window), so this costs no extra disk read; it only reframes the time axis to the last
        # `plotOpts.tailSeconds` simulated seconds, where the converged spread is legible.
        tailSeconds = runConfig["plotOpts"].get("tailSeconds", 100.0)
        tailSrc = goodTraces or {o: traces[o][finiteRow] for o in obsTargeted if o in (traces or {})}

        if tailSrc and plotT is not None:
            tAll = np.asarray(plotT)
            tailMask = tAll >= (tAll[-1] - tailSeconds)
            libPlots.plotCalibrationConvergence(
                {o: v[:, tailMask] for o, v in tailSrc.items()},
                traceT=tAll[tailMask], targets=targetsByObs, offsets=offsetsByObs,
                paramForObs=paramForObs,
                title=f"Calibration convergence -- last {tailSeconds:g}s "
                      f"({titleScope}, {simulationParams['solver']['type']})")
            plt.show()
        else:
            print("no traces / time axis to zoom -- run the convergence plot cell above first")
# endregion

## Parameter convergence plot

<details>
<summary>Companion to the observation plot above, for the **swept calibration parameters**. One panel</summary>

per parameter, overlaying each converged member's parameter trajectory (jet) with the **LHS
sampling edges** `[min, max]` as dotted lines — showing how tightly the population settles
inside the range it was sampled from. No target line (parameters have no fixed target) and no
legend.

</details>

In [ ]:
# region -> calibration convergence: one panel per swept parameter
if runConfig["plot"]:
    if good.sum() > 0:
        # --- calibration convergence: one panel per swept parameter --------------------------------
        # Each panel: the per-run parameter trajectory (jet) over the whole calibration and the LHS
        # sampling edges [min, max] (dotted grey). No legend, no target line (parameters have no fixed
        # target). Same source order as the observation plot: raw_coarse (instant) -> full raw
        # (strided) -> in-memory window.
        import h5py

        rangesByParam = {p: tuple(bounds[j]) for j, p in enumerate(param_names)}                   # edges
        goodIdx = np.where(finiteRow)[0]
        targetPoints = runConfig["plotOpts"]["targetPoints"] # strided-decimation target for the full-raw fallback

        paramTraces, plotT, titleScope = {}, traceT, "converged window"
        if saveRaw and runConfig["output"]["save"]:
            with h5py.File(outPath, "r") as f:
                if "raw_coarse" in f:                                   # fast path
                    src, tkey, ds, titleScope = "raw_coarse", "raw_coarse_time", 1, "whole calibration"
                else:                                                   # slow fallback (full raw)
                    src, tkey = "raw", "raw_time"
                    ds, titleScope = max(1, f["raw"].shape[1] // targetPoints), "whole calibration (full raw)"
                sig = list(f["raw_signal_names"].asstr()[:])
                names = [p for p in param_names if p in sig]
                block = f[src][goodIdx, ::ds, :]                        # good runs only, one pass
                plotT = f[tkey][::ds]
            paramTraces = {p: block[:, :, sig.index(p)] for p in names}
        else:
            paramTraces = {p: traces[p][finiteRow] for p in param_names if p in (traces or {})}

        libPlots.plotCalibrationConvergence(
            paramTraces, traceT=plotT, ranges=rangesByParam,
            paramForObs=None, showLegend=False,
            title=f"Calibration convergence -- parameters ({titleScope}, {simulationParams['solver']['type']})")
        plt.show()
# endregion

## Parameter convergence plot — tail zoom

<details>
<summary>The same parameter panels, zoomed to the last `plotOpts.tailSeconds` simulated seconds.</summary>

Shows where each swept parameter actually settles once the gains have done their work, at a
resolution the whole-calibration view can't give. Same traces as the cell above (no extra disk
read), masked to the tail window; the dotted LHS `[min, max]` edges are omitted so the y-axis
frames the converged band rather than the sampling range. Window knob:
`runConfig["plotOpts"]["tailSeconds"]`.

</details>

In [ ]:
# region -> parameter convergence (zoom): swept parameters over the last `plotOpts.tailSeconds`
if runConfig["plot"]:
    if good.sum() > 0:
        # --- same parameter traces as the plot above, masked to the END of the calibration -----
        # Reuses `paramTraces` / `plotT` from the cell above (no extra disk read). The LHS edge
        # lines are dropped here on purpose: axhlines at [min, max] expand the y-autoscale and
        # would squash the tail flat -- each panel frames its own converged band instead.
        tailSeconds = runConfig["plotOpts"].get("tailSeconds", 100.0)

        if paramTraces and plotT is not None:
            tAll = np.asarray(plotT)
            tailMask = tAll >= (tAll[-1] - tailSeconds)
            libPlots.plotCalibrationConvergence(
                {p: v[:, tailMask] for p, v in paramTraces.items()},
                traceT=tAll[tailMask],
                paramForObs=None, showLegend=False,
                title=f"Calibration convergence -- parameters, last {tailSeconds:g}s "
                      f"({titleScope}, {simulationParams['solver']['type']})")
            plt.show()
        else:
            print("no parameter traces / time axis to zoom -- run the parameter plot cell above first")
# endregion

## Run timings

<details>
<summary>Wall-clock cost of the batched sweep, loaded from the population file (`timingMeta`).</summary>

Because the whole population is one vmapped solve, the meaningful numbers are the
**total wall** for `nrModels` calibrations and the **amortized** seconds-per-calibration
(total ÷ N) — there is no per-run distribution to plot. Directly supports the reviewer's
timing ask; cross-file CPU-vs-GPU and scaling-vs-N comparisons live in the dedicated
timing-analysis notebook.

</details>

In [ ]:
# region -> batched timing summary (total wall + amortized per-calibration)
if runConfig["plot"]:
    # --- batched timing summary (total wall + amortized per-calibration) -----------------
    if timingMeta:
        totalWall = timingMeta.get("total_wall")
        n = timingMeta.get("nrModels") or pop["nrModels"]
        timingSummary = pd.DataFrame([{
            "device":          timingMeta.get("device", "?"),
            "precision":       timingMeta.get("precision", "?"),
            "solver":          timingMeta.get("solver", "?"),
            "dt":              timingMeta.get("dt"),
            "runTime":         timingMeta.get("runTime"),
            "nrModels":        n,
            "total_wall_s":    totalWall,
            "amortized_s/run": (totalWall / n) if (totalWall and n) else None,
        }])
        display(timingSummary)
    else:
        print("no timings in file (Phase 1 ran with output.save=False)")
# endregion

## (Deferred) NN training set

<details>
<summary>The population file is the input to the inverse-problem calibration NN (X = observations,</summary>

y = parameters). Deferred this round; the hook is:

```python
from library.hdf5 import schema_pop, schema_calib
arr, state_names = schema_pop.final_states_array(outPath)
training = schema_calib.build_training_set(
    arr, state_names, observation_keys=observations, param_keys=param_names)
schema_calib.save_calibration_run("notebookData/calibration_hr_test.h5", **training)
```

Note: this needs the observation values stored as columns in `final_states`; the current run
loop stores observations only as `runs/{id}/raw` traces. Wire the steady-state observation
values into the `final_state` row (or a parallel table) before enabling this.

</details>

In [ ]:
# region -> release GPU memory
# ---- release GPU memory --------------------------------------------------------------
# Drop references to the big result arrays, clear JAX's compiled caches, and (with
# XLA_PYTHON_CLIENT_ALLOCATOR=platform from the device cell) hand the VRAM back to the
# driver -- no kernel restart needed. A kernel restart is still the guaranteed full reset.
import gc
for _v in ("batch", "results", "finalStates", "traces", "traceT", "obsMatrix",
           "prep", "layout", "sampled_params", "finiteRow", "fsArr", "goodTraces", "block",
           "runWall", "lastRow"):
    globals().pop(_v, None)
jax.clear_caches()
gc.collect()
try:
    used = sum((d.memory_stats() or {}).get("bytes_in_use", 0) for d in jax.devices())
    print(f"GPU bytes in use after cleanup: {used/1e6:.0f} MB (restart kernel for a full reset)")
except Exception as e:
    print("memory_stats() unavailable on this device:", e)
# endregion